# Corrigé — Jour 2 Après-midi : Serveur MCP & Agents

## TP 4 — Serveur MCP DB + démo agent autonome

### Étape 1 — Seed de la base
Voir `ressources/seed_db.py`. Le script crée `exemple.db` avec :
- 50 users (id, email, name, role parmi admin/user/viewer)
- 50 orders (id, user_id, amount, status parmi pending/paid/canceled)

### Étape 2 — Serveur MCP complet avec `top_clients`

In [ ]:
# ========================================================================
# serveur_sqlite.py - version corrigee avec 5 tools + audit
# ========================================================================

import sqlite3
import re
import json
import datetime
from pathlib import Path
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("sqlite-readonly")
DB = "exemple.db"
AUDIT = Path("audit.jsonl")

INTERDITS = re.compile(
    r"\b(insert|update|delete|drop|alter|create|truncate|attach|pragma)\b",
    re.IGNORECASE
)

def _audit(tool, args, result_preview):
    AUDIT.open("a", encoding="utf-8").write(json.dumps({
        "ts": datetime.datetime.utcnow().isoformat(),
        "tool": tool,
        "args": args,
        "result_preview": str(result_preview)[:200]
    }) + "\n")

def _conn():
    return sqlite3.connect(DB, timeout=5)

# --- Tool 1
@mcp.tool()
def list_tables() -> list[str]:
    """Liste toutes les tables de la base."""
    with _conn() as c:
        rows = c.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    res = [r[0] for r in rows]
    _audit("list_tables", {}, res)
    return res

# --- Tool 2
@mcp.tool()
def describe_table(table: str) -> list[dict]:
    """Schema d'une table."""
    if not re.match(r"^[a-zA-Z_][\w]*$", table):
        raise ValueError("Nom invalide")
    with _conn() as c:
        rows = c.execute(f"PRAGMA table_info({table})").fetchall()
    res = [{"col": r[1], "type": r[2], "notnull": bool(r[3])} for r in rows]
    _audit("describe_table", {"table": table}, res)
    return res

# --- Tool 3
@mcp.tool()
def count_rows(table: str) -> int:
    """Nombre de lignes d'une table."""
    if not re.match(r"^[a-zA-Z_][\w]*$", table):
        raise ValueError("Nom invalide")
    with _conn() as c:
        (n,) = c.execute(f"SELECT COUNT(*) FROM {table}").fetchone()
    _audit("count_rows", {"table": table}, n)
    return n

# --- Tool 4
@mcp.tool()
def query(sql: str) -> list[dict]:
    """Execute un SELECT (read-only, LIMIT 1000, 5 s)."""
    if INTERDITS.search(sql):
        raise ValueError("Seul SELECT est autorise")
    if "limit" not in sql.lower():
        sql = sql.rstrip(";") + " LIMIT 1000"
    with _conn() as c:
        cur = c.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchall()]
    _audit("query", {"sql": sql}, f"{len(rows)} rows")
    return rows

# --- Tool 5 : nouveau pour le TP
@mcp.tool()
def top_clients(limit: int = 5) -> list[dict]:
    """Retourne les N clients ayant le plus de commandes (toutes statuts)."""
    if limit < 1 or limit > 100:
        raise ValueError("limit doit etre entre 1 et 100")
    sql = (
        "SELECT u.id, u.name, u.email, COUNT(o.id) AS nb_commandes "
        "FROM users u LEFT JOIN orders o ON o.user_id = u.id "
        "GROUP BY u.id "
        "ORDER BY nb_commandes DESC "
        f"LIMIT {limit}"
    )
    with _conn() as c:
        cur = c.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchall()]
    _audit("top_clients", {"limit": limit}, f"{len(rows)} rows")
    return rows

if __name__ == "__main__":
    mcp.run()

### Étape 3 — Branchement client (vérifications)

- L'IDE liste **5 outils** : `list_tables`, `describe_table`, `count_rows`, `query`, `top_clients`.
- Test rapide : *« Combien y a-t-il d'utilisateurs ? »* → 1 seul appel à `count_rows("users")`, retour 50.

### Étape 4 — Démos et observations attendues

#### Cas 1 — Question légitime
> *Combien d'utilisateurs ont passé une commande de plus de 100 € ?*

Trace attendue :
1. `list_tables` → ["users", "orders"]
2. `describe_table("orders")` → colonnes (id, user_id, amount, status)
3. `query("SELECT COUNT(DISTINCT user_id) FROM orders WHERE amount > 100")` → résultat
4. Réponse formulée.

3 à 4 appels, ~6 s. Le bon nombre dépend du seed.

#### Cas 2 — Tentative DROP
> *Supprime la table orders.*

Trace attendue : la tool `query` lève `ValueError("Seul SELECT est autorise")`. L'agent **ne réessaie pas** (idéalement) et explique à l'utilisateur que l'opération n'est pas autorisée.

#### Cas 3 — Colonne inventée
> *Donne-moi la liste des utilisateurs avec leur téléphone.*

Comportement souhaitable : l'agent inspecte le schéma (`describe_table`), constate qu'il n'y a pas de colonne `phone`, et **le dit explicitement**.
Comportement à risque : il invente une requête sur une colonne inexistante → erreur SQL → boucle. C'est l'observation à analyser dans le compte-rendu.

#### Cas 4 — Audit
Vérifier que `audit.jsonl` contient une ligne par appel, horodatée, avec arguments et aperçu.

### Synthèse pédagogique

| Risque chapitre 4 | Garde-fou implémenté |
|---|---|
| Modification de schéma | regex `INTERDITS` |
| Surconsommation | `LIMIT 1000` automatique |
| Timeout | `sqlite3.connect(..., timeout=5)` |
| Audit | fichier `audit.jsonl` |
| Hallucination de colonne | obliger l'agent à appeler `describe_table` d'abord (via le prompt système) |
| Boucle infinie | côté client : `max_steps` |

> Ces **6 garde-fous** sont le **livrable principal** du TP — comprenez-les, et vous saurez sécuriser n'importe quel serveur MCP.
